In [1]:
import json
import random
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
random.seed(42)

CADEC_PATH = "../data/raw/CADEC/data/CADEC.v2/cadec"
TEXT_DIR   = Path(CADEC_PATH) / "text"
ORIG_DIR   = Path(CADEC_PATH) / "original"
MEDDRA_DIR = Path(CADEC_PATH) / "meddra"
SCT_DIR    = Path(CADEC_PATH) / "sct"

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# MedDRA code → severity label
# Based on standard MedDRA SOC hierarchy
# Top codes covering ~80% of CADEC dataset
SEVERITY_MAP = {
    # Life-threatening
    "10013968": "life_threatening",  # Dyspnoea (shortness of breath)
    "10003206": "life_threatening",  # Anaphylactic reaction
    "10000503": "life_threatening",  # Acute hepatic failure
    "10019491": "life_threatening",  # Hepatic failure
    "10042434": "life_threatening",  # Sudden death
    "10011906": "life_threatening",  # Death
    "10007196": "life_threatening",  # Cardiac arrest
    "10038290": "life_threatening",  # Rhabdomyolysis
    "10046219": "life_threatening",  # Thrombocytopenia
    "10002272": "life_threatening",  # Anaphylactic shock

    # Serious
    "10028411": "serious",           # Myalgia (muscle pain)
    "10003239": "serious",           # Arthralgia (joint pain)
    "10012378": "serious",           # Depression
    "10028350": "serious",           # Muscle weakness
    "10003549": "serious",           # Asthenia (weakness)
    "10016256": "serious",           # Fatigue
    "10029829": "serious",           # Numbness
    "10035067": "serious",           # Paraesthesia (tingling)
    "10043882": "serious",           # Tinnitus
    "10027175": "serious",           # Memory impairment
    "10033371": "serious",           # Pain in extremity
    "10033473": "serious",           # Pain in limb
    "10033432": "serious",           # Pain in hip
    "10033430": "serious",           # Pain in hand
    "10047810": "serious",           # Weight increased
    "10001949": "serious",           # Alopecia (hair loss)
    "10022437": "serious",           # Insomnia
    "10013573": "serious",           # Elevated liver enzymes
    "10043890": "serious",           # Tiredness
    "10011301": "serious",           # Dizziness
    "10003993": "serious",           # Back pain
    "10064774": "serious",           # Myopathy
    "10033376": "serious",           # Pain on walking

    # Non-serious
    "10019211": "non_serious",       # Headache
    "10028813": "non_serious",       # Nausea
    "10012735": "non_serious",       # Diarrhoea
    "10028294": "non_serious",       # Muscle cramps
    "10016766": "non_serious",       # Flatulence (gas)
    "10025482": "non_serious",       # Malaise
    "10013649": "non_serious",       # Drowsiness
    "10005886": "non_serious",       # Blurred vision
    "10046318": "non_serious",       # Upset stomach
    "10042076": "non_serious",       # Stomach pain
    "10002959": "non_serious",       # Aphthous ulcer (canker sore)
    "10046883": "non_serious",       # Vaginal bleeding
    "10056819": "non_serious",       # GI discomfort
    "10048721": "non_serious",       # Upper GI sounds
    "10019326": "non_serious",       # Heartburn
    "10033494": "non_serious",       # Pharyngeal pain (sore throat)
    "10042645": "non_serious",       # Swallowing difficulty (mild)
    "10033407": "non_serious",       # Hunger
}

In [12]:
def get_severity_from_codes(meddra_codes):
    if not meddra_codes:
        return "non_serious"  # no ADRs reported = non serious
    order = ["life_threatening", "serious", "non_serious"]
    severities = [SEVERITY_MAP.get(code, "serious") for code in meddra_codes]
    for level in order:
        if level in severities:
            return level
    return "serious"

print(f"SEVERITY_MAP has {len(SEVERITY_MAP)} entries")
print("Done.")

SEVERITY_MAP has 51 entries
Done.


In [13]:
def parse_ann(ann_path):
    annotations = []
    if not ann_path.exists():
        return annotations
    with open(ann_path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) < 3:
                continue
            meta = parts[1].split()
            ann_type = meta[0]
            text = parts[2]
            annotations.append({'type': ann_type, 'text': text})
    return annotations

In [14]:
def load_cadec_records():
    records = []
    for txt_file in sorted(TEXT_DIR.glob('*.txt')):
        stem = txt_file.stem
        drug_name = stem.split('.')[0]

        with open(txt_file, encoding='utf-8', errors='ignore') as f:
            report_text = f.read().strip()

        orig_anns   = parse_ann(ORIG_DIR   / f"{stem}.ann")
        meddra_anns = parse_ann(MEDDRA_DIR / f"{stem}.ann")

        adrs = [a['text'] for a in orig_anns if a['type'] == 'ADR']
        meddra_codes = [
            a['type'] for a in meddra_anns
            if not a['type'].startswith('CONCEPT')
        ]
        has_unmapped = any(
            'CONCEPT_LESS' in a['type'] for a in meddra_anns
        )

        severity = get_severity_from_codes(meddra_codes)

        # causality: unmapped = ambiguous = 0.5, else 0.8
        causality = 0.5 if has_unmapped else (0.8 if adrs else 0.1)

        # escalate if serious or worse
        escalate = severity in ["serious", "life_threatening", "fatal"]

        # recommended action based on severity
        action_map = {
            "non_serious":      "monitor_only",
            "serious":          "request_followup",
            "life_threatening": "expedited_review",
            "fatal":            "urgent_regulatory_notification",
        }

        records.append({
            "report_id":          stem,
            "drug_name":          drug_name,
            "report_text":        report_text,
            "reported_symptoms":  adrs,
            "meddra_codes":       meddra_codes,
            "has_unmapped":       has_unmapped,
            "adr_count":          len(adrs),
            "gt_severity":        severity,
            "gt_causality":       causality,
            "gt_escalate":        escalate,
            "gt_recommended_action": action_map[severity],
        })
    return records

In [15]:
cadec_records = load_cadec_records()
print(f"Loaded {len(cadec_records)} CADEC records")

# Quick check
from collections import Counter
sev_dist = Counter(r['gt_severity'] for r in cadec_records)
print("Severity distribution:", dict(sev_dist))

Loaded 1250 CADEC records
Severity distribution: {'non_serious': 192, 'life_threatening': 16, 'serious': 1042}


### task1_dataset.json

In [16]:
def build_task1(df_ade, n_each=75):
    """150 ADE Corpus sentences, balanced 75 ADE / 75 non-ADE."""
    ade     = df_ade[df_ade['label'] == 1].sample(n_each, random_state=42)
    non_ade = df_ade[df_ade['label'] == 0].sample(n_each, random_state=42)
    combined = pd.concat([ade, non_ade]).sample(frac=1, random_state=42).reset_index(drop=True)

    records = []
    for _, row in combined.iterrows():
        severity = "serious" if row['label'] == 1 else "non_serious"
        records.append({
            "episode_id":         f"task1_{len(records):03d}",
            "report_text":        row['text'],
            "drug_name":          "UNKNOWN",
            "patient_age":        None,
            "reported_symptoms":  [],
            "prior_actions_taken": [],
            "gt_severity":        severity,
            "gt_causality":       0.9 if row['label'] == 1 else 0.1,
            "gt_escalate":        row['label'] == 1,
            "gt_recommended_action": "request_followup" if row['label'] == 1 else "monitor_only",
        })

    path = OUTPUT_DIR / "task1_dataset.json"
    with open(path, 'w') as f:
        json.dump(records, f, indent=2)

    print(f"task1_dataset.json — {len(records)} records saved to {path}")
    return records

# Load ADE corpus
df_ade = pd.read_parquet("../data/raw/ADE_CORPUS/train-00000-of-00001.parquet")
task1 = build_task1(df_ade)

# Sanity check
print("Sample record:")
print(json.dumps(task1[0], indent=2))

task1_dataset.json — 150 records saved to ..\data\processed\task1_dataset.json
Sample record:
{
  "episode_id": "task1_000",
  "report_text": "A 61-year-old man with early diffuse cutaneous scleroderma with myositis and progressive interstitial pneumonia developed generalized erythema with high fever 3 weeks after taking sulfamethoxazole/trimethoprim.",
  "drug_name": "UNKNOWN",
  "patient_age": null,
  "reported_symptoms": [],
  "prior_actions_taken": [],
  "gt_severity": "serious",
  "gt_causality": 0.9,
  "gt_escalate": true,
  "gt_recommended_action": "request_followup"
}


### task2_dataset.json

In [18]:
def build_task2(cadec_records, n=100):
    complex_records = [r for r in cadec_records if r['adr_count'] >= 3]
    non_serious     = [r for r in cadec_records if r['gt_severity'] == 'non_serious' and r['adr_count'] > 0]

    random.shuffle(complex_records)
    random.shuffle(non_serious)

    # 70 complex serious + 30 non-serious
    selected = complex_records[:70] + non_serious[:30]
    random.shuffle(selected)

    records = []
    for r in selected:
        records.append({
            "episode_id":            f"task2_{len(records):03d}",
            "report_text":           r['report_text'],
            "drug_name":             r['drug_name'],
            "patient_age":           None,
            "reported_symptoms":     r['reported_symptoms'],
            "prior_actions_taken":   [],
            "gt_severity":           r['gt_severity'],
            "gt_causality":          r['gt_causality'],
            "gt_escalate":           r['gt_escalate'],
            "gt_recommended_action": r['gt_recommended_action'],
        })

    path = OUTPUT_DIR / "task2_dataset.json"
    with open(path, 'w') as f:
        json.dump(records, f, indent=2)

    print(f"task2_dataset.json — {len(records)} records saved to {path}")
    sev = Counter(r['gt_severity'] for r in records)
    esc = Counter(r['gt_escalate'] for r in records)
    print("Severity distribution:", dict(sev))
    print("Escalate distribution:", dict(esc))
    return records

task2 = build_task2(cadec_records)

print("\nSample record:")
print(json.dumps(task2[0], indent=2))

task2_dataset.json — 100 records saved to ..\data\processed\task2_dataset.json
Severity distribution: {'serious': 65, 'life_threatening': 4, 'non_serious': 31}
Escalate distribution: {True: 69, False: 31}

Sample record:
{
  "episode_id": "task2_000",
  "report_text": "Cramps in my feet, pain in the lower part of my knees, fatigue, mental fog.\nDuring the three years that I was taking Lipitor, my cholesterol averaged as follows (n = 5) all units mg/dL: Total Cholesterol - 140 HDL - 38 LDL - 74 Triglycerides - 139 Total/HDL Ratio - 3.67\nAfter stopping Lipitor and all other statins in 2010: Total Cholesterol - 218 HDL - 34 LDL - 184 Triglycerides - 167 Total/HDL ratio - 6.4\nAfter switching to 10 mg Simvistatin: Total Cholesterol - 136 HDL - 38 LDL - 70 Triglycerides - 138 Total/HDL ratio - 3.58\nI take 1/2 of a 20 mg generic Simvistatin pill daily.\nVery little side effects, cheaper, and does the same job as Lipitor.",
  "drug_name": "LIPITOR",
  "patient_age": null,
  "reported_sympto

### task3_dataset.json

In [21]:
def build_task3(cadec_records, n_clusters=30):
    lipitor = [r for r in cadec_records if r['drug_name'] == 'LIPITOR' and r['adr_count'] > 0]
    random.shuffle(lipitor)

    from collections import defaultdict
    code_groups = defaultdict(list)
    for r in lipitor:
        if r['meddra_codes']:
            primary_code = r['meddra_codes'][0]
            code_groups[primary_code].append(r)

    valid_groups = {k: v for k, v in code_groups.items() if len(v) >= 5}
    print(f"Valid groups (5+ records): {len(valid_groups)}")

    clusters = []
    used_ids = set()

    # TRUE SIGNAL — serious/life-threatening codes, 15 clusters
    signal_codes = [
        k for k, v in valid_groups.items()
        if SEVERITY_MAP.get(k, 'serious') in ['serious', 'life_threatening']
    ]
    random.shuffle(signal_codes)

    for code in signal_codes:
        if len(clusters) >= 15:
            break
        candidates = [r for r in valid_groups[code] if r['report_id'] not in used_ids]
        if len(candidates) < 5:
            continue
        selected = candidates[:5]
        for r in selected:
            used_ids.add(r['report_id'])
        clusters.append({
            "episode_id":            f"task3_{len(clusters):03d}",
            "drug_name":             "LIPITOR",
            "is_signal":             True,
            "signal_code":           code,
            "gt_escalate":           True,
            "gt_recommended_action": "signal_team_review",
            "reports": [{
                "report_text":       r['report_text'],
                "reported_symptoms": r['reported_symptoms'],
                "drug_name":         r['drug_name'],
            } for r in selected]
        })

    # NOISE — pick 5 random records from different codes (mixed, non-coherent)
    remaining = [r for r in lipitor if r['report_id'] not in used_ids]
    random.shuffle(remaining)

    # chunk remaining into groups of 5
    noise_groups = [remaining[i:i+5] for i in range(0, len(remaining), 5) if len(remaining[i:i+5]) == 5]

    for group in noise_groups:
        if len(clusters) >= 30:
            break
        for r in group:
            used_ids.add(r['report_id'])
        clusters.append({
            "episode_id":            f"task3_{len(clusters):03d}",
            "drug_name":             "LIPITOR",
            "is_signal":             False,
            "signal_code":           None,
            "gt_escalate":           False,
            "gt_recommended_action": "monitor_only",
            "reports": [{
                "report_text":       r['report_text'],
                "reported_symptoms": r['reported_symptoms'],
                "drug_name":         r['drug_name'],
            } for r in group]
        })

    random.shuffle(clusters)
    for i, c in enumerate(clusters):
        c['episode_id'] = f"task3_{i:03d}"

    path = OUTPUT_DIR / "task3_dataset.json"
    with open(path, 'w') as f:
        json.dump(clusters, f, indent=2)

    print(f"task3_dataset.json — {len(clusters)} clusters saved to {path}")
    signals = Counter(c['is_signal'] for c in clusters)
    print("Signal distribution:", dict(signals))
    return clusters

task3 = build_task3(cadec_records)

print("\nSample cluster:")
sample = task3[0].copy()
sample['reports'] = sample['reports'][:1]  # show just first report
print(json.dumps(sample, indent=2))

Valid groups (5+ records): 35
task3_dataset.json — 30 clusters saved to ..\data\processed\task3_dataset.json
Signal distribution: {False: 15, True: 15}

Sample cluster:
{
  "episode_id": "task3_000",
  "drug_name": "LIPITOR",
  "is_signal": false,
  "signal_code": null,
  "gt_escalate": false,
  "gt_recommended_action": "monitor_only",
  "reports": [
    {
      "report_text": "HORRIBLE JOINT PAIN IN ALL MAJOR JOINTS, INCLUDING PROGRESSING TO ALMOST UNBEARABLE MUSCLE PAIN IN ALL MUSCLES IN BOTH UPPER AND LOWER EXTREMITIES, HAIR FALLING OUT, PRESCRIBED DARVACET FOR THE PAIN AND WAS ALMOST DIAGNOSED WITH FIBROMYALGIA, SIMILAR SYMPTOMS.\nMY MOTHER HAS HAD THE SAME SYMPTOMS, ALSO WAS ON LIPITOR AND WAS RECENTLY TAKEN OFF BY HER NURSE PRACTITONER.\nIF YOU ARE TAKING THIS MEDICATION CLOSELY WATCH FOR JOINT OR MUSCLE PAIN AND IMMEDIATELY SEE YOUR PRESCRIBING PHYSICIAN FOR A DIAGNOSIS IF YOU WANT RELIEF.\nI DON'T RECOMMEND THIS MEDICATION FOR THE LOWERING OF CHOLESTEROL.",
      "reported_symp

In [22]:
# Final check — all 3 datasets
for name, path in [
    ("task1", OUTPUT_DIR / "task1_dataset.json"),
    ("task2", OUTPUT_DIR / "task2_dataset.json"),
    ("task3", OUTPUT_DIR / "task3_dataset.json"),
]:
    with open(path) as f:
        data = json.load(f)
    print(f"{name}: {len(data)} records — {path}")

print("\nAll 3 datasets ready.")

task1: 150 records — ..\data\processed\task1_dataset.json
task2: 100 records — ..\data\processed\task2_dataset.json
task3: 30 records — ..\data\processed\task3_dataset.json

All 3 datasets ready.
